# Calculating a Shock Extreme Typical Meteorological Year -- Methodology Walkthrough

Extreme meteorological years (XMYs) are intended to capture extreme events, like heat waves and cold snaps. We define two types of XMYs:

1. persistence: sustained extremes (ex: an unusualy hot summer)
2. shock: system shocks, weather events (ex: heat waves, cold snaps)

This notebook covers to the steps to generate shock XMYs.

The shock XMY methodology utilizes the following small changes to the TMY methodology to get at cold and hot extremes:
1. Modifies the Finkelstein-Schafer statistic calculation to use the relative, rather than absolute, difference. This preserves the direction of the difference.
2. When selecting the candidate months, select values that differ MOST from the climatological CDF in the negative direction for cold shock events, and the positive direction for hot shock events.

### Step 0: Set-up

Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [72]:
import pandas as pd
import xarray as xr
import numpy as np
import hvplot.xarray
from importlib.resources import files
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt

import climakitae as ck
from climakitae.core.constants import UNSET
from climakitae.core.data_export import write_tmy_file

# Initialize ClimateData object
cd = ck.ClimateData(verbosity=-1)

### Step 1: Grab and process all required input data
We use the TMY3 method to guide variable selection. The method selects a "typical" month based on ten daily variables: max, min, and mean air and dew point temperatures, max and mean wind speed, global irradiance and direct irradiance.

#### Step 1a: Select location of interest
XMYs are calculated for a specific location of interest, like a building or power plant. Here, we will use a known weather station location, via their latitude and longitude to extract the data that we need to calculate the XMY. In the example below, we will look specifically at Los Angeles International Airport, but will note in the code below how you can provide your own location coordinates too.

In [73]:
# read in station file of CA HadISD stations
stn_filepath_s3 = "s3://cadcat/hadisd/hadisd_stations.csv"
stn_file = pd.read_csv(stn_filepath_s3, index_col=[0])
stn_file.head(5)

,state,station,city,ID,LAT_Y,LON_X,station id,elevation
0,CA,Bakersfield Meadows Field (KBFL),Bakersfield,KBFL,35.43424,-119.05524,72384023155,149.3
1,CA,Blythe Asos (KBLH),Blythe,KBLH,33.61876,-114.71451,74718823158,120.4
2,CA,Burbank-Glendale-Pasadena Airport (KBUR),Burbank,KBUR,34.19966,-118.36543,72288023152,222.7
3,CA,Needles Airport (KEED),Needles,KEED,34.76783,-114.61842,72380523179,270.6
4,CA,Fresno Yosemite International Airport (KFAT),Fresno,KFAT,36.77999,-119.72016,72389093193,101.9


In [74]:
# grab airport
stn_name = "Ontario International Airport (KONT)"
stn_code = stn_file.loc[stn_file["station"] == stn_name]["station id"].item()
one_station = stn_file.loc[stn_file["station"] == stn_name]
stn_lat = one_station.LAT_Y.item()
stn_lon = one_station.LON_X.item()
stn_state = one_station.state.item()

# Output results
print("station coordinates:", (stn_lat, stn_lon))
print("station code:", stn_code)

station coordinates: (34.067, -117.65)
station code: 74704003102


Alternatively, you may want to provide your own location instead of one of the HadISD stations above. If so, uncomment the cell below by removing the `#` symbol and modifying the lines of code. Note, with custom locations, if an elevation is not provided, a default value of 0.0 m will be used. 

In [75]:
## provide your own location, via latitude and longitude coordinates
# stn_lat = YOUR_LAT_HERE
# stn_lon = YOUR_LON_HERE
# stn_state = 'YOUR_STATE_HERE'
# stn_name = 'YOUR_STATION_NAME_HERE'
# stn_code = 'custom'
# stn_elev = YOUR_ELEV_HERE # meters

#### Step 1b: Select time frame of interest
The second required input for generating a XMY dataset is the **time frame of interest**. The recommended minimum number of input years for a TMY dataset is 15-20 years worth of daily data; we will use 30 years to represent a standard climatological period. For data post-2014, we will utilize SSP 3-7.0, although scenario selection in the near-future is relatively independent. If calculating a TMY for the far-future, **carefully consider which scenario SSP to include**, as there will be **significant** differences present. 

We will also process the data for our designated station location (latitude, and longitude) at 3 km over the <span style="color:#FF0000">1990-2020 period</span> as an example. **Note**: An additional year (2021) is also loaded to pad the end of the dataset after converting to local time in the next few steps -- this is done for you when subsetting for the data. 

In [76]:
# selected reference period
start_year = 2005
end_year = 2020

#### Step 1c: Retrieve variables in catalog
It is important to note that not all models in the Cal-Adapt: Analytics Engine have the solar variables critical for XMY file generation - in fact, only 4 do! We will carefully subset our variables to ensure that the same 4 models are selected for consistency. 

Lastly, because the dynamically downscaled WRF data in the Cal-Adapt: Analytics Engine is in UTC time, we'll convert to the timezone of the station we've selected. This is particularly important for the timing of solar radiation max and min, which is a critical piece of a TMY. The handy `convert_to_local_time` function corrects for this, and ensures that all data are within the same daily timestamp.

In [77]:
# selected models
data_models = [
    "wrf_ucla_taiesm1_ssp370_r1i1p1f1",
    "wrf_ucla_ec-earth3_ssp370_r1i1p1f1",
    "wrf_ucla_miroc6_ssp370_r1i1p1f1",
    "wrf_ucla_mpi-esm1-2-hr_ssp370_r3i1p1f1",
]

# map variable names to descriptive names and units
var_info = {
    "t2": {"long_name": "Air temperature at 2m", "units": "degC"},
}

Now that we have set up the model list, let's start retrieving data! We will need to calculate summary statistics and reduce to daily timescales for the following variables:

In [ ]:
t2_ds = (
    cd.catalog("cadcat")
    .variable("t2")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "convert_units": "degC",
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

In [ ]:
# Subset simulations and convert to DataArray
t2_ds = t2_ds.sel(sim=data_models, time=slice(str(start_year), str(end_year))).t2

# Load into memory using dask progress bar
with ProgressBar():
    t2_ds.load()

In [ ]:
# max air temp
max_airtemp_data = t2_ds.resample(time="1D").max()  # daily max air temp
max_airtemp_data.name = "Daily max air temperature"  # rename for clarity

# min air temp
min_airtemp_data = t2_ds.resample(time="1D").min()  # daily min air temp
min_airtemp_data.name = "Daily min air temperature"  # rename for clarity

# mean air temp
mean_airtemp_data = t2_ds.resample(time="1D").mean()  # daily mean air temp
mean_airtemp_data.name = "Daily mean air temperature"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del t2_ds

Retrieve and calculate max and mean wind speed:

In [ ]:
# wind speed
ws_ds = (
    cd.catalog("cadcat")
    .variable("wind_speed_10m")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

In [ ]:
# Subset simulations and convert to DataArray
ws_ds = ws_ds.sel(
    sim=data_models, time=slice(str(start_year), str(end_year))
).wind_speed_10m

# Load into memory using dask progress bar
with ProgressBar():
    ws_ds.load()

In [ ]:
# max wind speed
max_windspd_data = ws_ds.resample(time="1D").max()  # daily max wind speed
max_windspd_data.name = "Daily max wind speed"  # rename for clarity

# mean wind speed
mean_windspd_data = ws_ds.resample(time="1D").mean()  # daily mean wind speed
mean_windspd_data.name = "Daily mean wind speed"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del ws_ds

Retrieve and calculate max, min, and mean dew point temperature:

In [ ]:
# dew point temperature
dewpt_ds = (
    cd.catalog("cadcat")
    .variable("dew_point_2m")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_units": "degC",
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

In [ ]:

# Subset simulations and convert to DataArray
dewpt_ds = dewpt_ds.sel(
    sim=data_models, time=slice(str(start_year), str(end_year))
).dew_point_2m

# Load into memory using dask progress bar
with ProgressBar():
    dewpt_ds.load()

In [ ]:
# max dew point
max_dewpt_data = dewpt_ds.resample(time="1D").max()  # daily max dewpoint temp
max_dewpt_data.name = "Daily max dewpoint temperature"  # rename for clarity

# min dew point
min_dewpt_data = dewpt_ds.resample(time="1D").min()  # daily min dewpoint temp
min_dewpt_data.name = "Daily min dewpoint temperature"  # rename for clarity

# mean dew point
mean_dewpt_data = dewpt_ds.resample(time="1D").mean()  # daily mean dewpoint temp
mean_dewpt_data.name = "Daily mean dewpoint temperature"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del dewpt_ds

Next, retrieve global horizontal irradiance. GHI is within the Analytics Engine catalog at daily resolutions, but for the XMY methodology, we need to calculate the total accumulated GHI received over the course of the day, so we will retrieve hourly data instead and calculate the necessary information below. The same process is repeated for the direct normal irradiance. 

In [ ]:
# global irradiance
global_irradiance_ds = (
    cd.catalog("cadcat")
    .variable("swdnb")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

In [ ]:

# Subset simulations and convert to DataArray
global_irradiance_ds = global_irradiance_ds.sel(
    sim=data_models, time=slice(str(start_year), str(end_year))
).swdnb

# Load into memory using dask progress bar
with ProgressBar():
    global_irradiance_ds.load()

In [ ]:
# total global horizontal irradiance (accumulation of hourly data over a 24-hour period)
total_ghi_data = global_irradiance_ds.resample(time="1D").sum()
total_ghi_data.name = "Global horizontal irradiance"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del global_irradiance_ds

The same process is repeated for the direct normal irradiance.

In [ ]:
# direct normal irradiance
direct_irradiance_ds = (
    cd.catalog("cadcat")
    .variable("swddni")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

In [ ]:

# Subset simulations and convert to DataArray
direct_irradiance_ds = direct_irradiance_ds.sel(
    sim=data_models, time=slice(str(start_year), str(end_year))
).swddni

# Load into memory using dask progress bar
with ProgressBar():
    direct_irradiance_ds.load()

In [ ]:
# total direct normal irradiance (accumulation of hourly data over a 24-hour period)
total_dni_data = direct_irradiance_ds.resample(time="1D").sum()
total_dni_data.name = "Direct normal irradiance"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del direct_irradiance_ds

#### Step 1d: Load all variables
Now that we have all of our data retrieved and calculated, it is time to actually load the data into memory. Previously, xarray has lazily loaded the data, but we will actually grab it now.

In [ ]:
all_vars = xr.merge(
    [
        max_airtemp_data,
        min_airtemp_data,
        mean_airtemp_data,
        max_dewpt_data,
        min_dewpt_data,
        mean_dewpt_data,
        max_windspd_data,
        mean_windspd_data,
        total_ghi_data,
        total_dni_data,
    ]
)

### Step 2: Calculate cumulative distribution functions

The cumulative distribution function gives the proportion of values that are less than or equal to a specified value of the index. In this case, we want to identify months that differ most from the long-term climatology for each variable as possible, indicating months that are "extreme".  

#### Step 2a: Calculate long-term climatology CDFs for each index
First, we need to calculate the long-term climatology for each index for each month so we can establish the baseline pattern. 

In [78]:
# load saved all_vars eagerly
all_vars = xr.load_dataset("all_vars.nc")

In [79]:
def compute_cdf(da):
    """Compute the cumulative density function for an input DataArray"""
    da_np = da.values  # Get numpy array of values
    num_samples = 1024  # Number of samples to generate
    count, bins_count = np.histogram(  # Create a numpy histogram of the values
        da_np,
        bins=np.linspace(
            da_np.min(),  # Start at the minimum value of the array
            da_np.max(),  # End at the maximum value of the array
            num_samples,
        ),
    )
    cdf_np = np.cumsum(count / sum(count))  # Compute the CDF

    # Turn the CDF array into xarray DataArray
    # New dimension is the bin values
    cdf_da = xr.DataArray(
        [bins_count[1:], cdf_np],
        dims=["data", "bin_number"],
        coords={
            "data": ["bins", "probability"],
        },
    )
    cdf_da.name = da.name
    return cdf_da


def get_cdf_by_sim(da):
    # Group the DataArray by simulation
    return da.groupby("sim").map(compute_cdf)


def get_cdf_by_mon_and_sim(da):
    # Group the DataArray by month in the year
    return da.groupby("time.month").map(get_cdf_by_sim)


def get_cdf(ds):
    """Get the cumulative density function.

    Parameters
    -----------
    ds: xr.Dataset
        Input data to compute CDF for
    Returns
    -------
    xr.Dataset
    """
    return ds.map(get_cdf_by_mon_and_sim)

In [80]:
cdf_climatology = get_cdf(all_vars)

In the plot below, we'll display maximum air temperature to assess the climatological CDF pattern, but you can modify the variable here to one of your choosing to see the pattern too! Also select a different month by moving the slider bar to see the pattern throughout the year.

In [81]:
def plot_cdf(cdf_da: xr.DataArray) -> hvplot:
    """
    Plot CDF with simulations as a grouped overlay.

    Parameters
    ----------
    cdf_da : xr.DataArray
        Output of compute_cdf, with a 'data' dimension containing 'bins' and 'probability'.

    Returns
    -------
    hvplot
        CDF plot object.
    """
    ds = xr.merge(
        [
            cdf_da.sel(data="probability", drop=True).rename("probability"),
            cdf_da.sel(data="bins", drop=True).rename("bins"),
        ]
    )
    cdf_pl = ds.hvplot(
        "bins",
        "probability",
        by="sim",
        widget_location="bottom",
        grid=True,
        xlabel=f"{cdf_da.name} ({cdf_da.attrs['units']})",
        xlim=(ds["bins"].min().item(), ds["bins"].max().item()),
        ylabel="Probability (0-1)",
    )
    return cdf_pl

In [82]:
# Choose your desired variable
var = "Daily mean air temperature"

# Make plot
plot_cdf(cdf_climatology[var])

BokehModel(combine_events=True, render_bundle={'docs_json': {'ed1914a8-e4a1-4402-96b0-f584463613d5': {'version…

#### Step 2b: Calculate CDFs for each index for all months

Next, we will calculate CDF for each month and each variable, for which we ultimately will compare against climatology. For the individual months, we must also exclude the period of time during a major volcanic eruption (Pinatubo: June 1991 to December 1994) as the aerosols have an impact on solar variables. The cells below functions exclude this data from our data next. 

In [83]:
def get_cdf_monthly(ds):
    """Get the cumulative density function by unique mon-yr combos

    Parameters
    -----------
    ds: xr.Dataset
        Input data to compute CDF for
    Returns
    -------
    xr.Dataset
    """

    def get_cdf_mon_yr(da):
        return da.groupby("time.year").apply(get_cdf_by_mon_and_sim)

    return ds.apply(get_cdf_mon_yr)

In [84]:
cdf_monthly = get_cdf_monthly(all_vars)

Now we'll remove the volcanic years: 

In [85]:
# Remove the years for the Pinatubo eruption
cdf_monthly = cdf_monthly.where(
    (~cdf_monthly.year.isin([1991, 1992, 1993, 1994])), np.nan, drop=True
)

Like the climatology CDF figure above, let's check out the individual months next. You can modify the variable, and month-year to display too.

In [86]:
# Choose your desired variable
var = "Daily mean air temperature"

# Make the plot
plot_cdf(cdf_monthly[var])

BokehModel(combine_events=True, render_bundle={'docs_json': {'df3109d6-a3d0-4c1e-8c89-58d74ff080ec': {'version…

### Step 3: Select candidate months for consideration

Now that we have the distributions for the long-term climatology of our 30-year period, and the corresponding distribution for each month in that 30-year period, we need to assess how each individual month compares to the long-term climatology. In essence, we are looking for the years (i.e. year for each month) that most differ from the climatology distribution in the desired direction.

We have heard from utilities that air temperature is the single priority variable they need from XMYs. Instead of weighting to identify which year is selected, then, we select years based on air temperature extremes. In particular, we use minimum air temperature for selecting cold shock XMY years, and maximum temperature for selecting hot shock XMY years. We use the deviation from the median temperature - that is, for CDF=0.5 - to determine the "worst" year for each month. This ensures that the years with the most extreme temperature shocks are selected.

In [ ]:
def find_hot_cold_extreme_from_median(
    sub_year: xr.DataArray,
    sub_clim: xr.DataArray,
    target: float = 0.5,
    extreme: str = "cold",  # "cold" or "hot"
):
    """
    Identify hottest or coldest year based on deviation from the median (CDF=0.5)
    temperature value across years.

    Parameters
    ----------
    sub_year : xr.DataArray
        dims: (year, data, bin_number)

    sub_clim : xr.DataArray
        climatology with same structure

    target : float
        CDF level (default 0.5)

    extreme : str
        "cold" -> pick minimum deviation
        "hot"  -> pick maximum deviation

    Returns
    -------
    results : list
        median values per year

    worst_year : scalar
        identified extreme year
    """

    # -----------------------------
    # Step 1: climatology median (p50 reference)
    # -----------------------------
    clim_prob = sub_clim.sel(data="probability")
    clim_bins = sub_clim.sel(data="bins")

    clim_idx = np.abs(clim_prob - target).argmin(dim="bin_number")
    clim_05 = clim_bins.isel(bin_number=clim_idx).values

    # -----------------------------
    # Step 2: per-year median extraction
    # -----------------------------
    yr_prob = sub_year.sel(data="probability")
    yr_bins = sub_year.sel(data="bins")

    idx_list = (
        np.abs(yr_prob - target)
        .argmin(dim="bin_number")
        .values
        .tolist()
    )

    results = []

    years = sub_year["year"].values

    for i, yr in enumerate(years):
        idx = idx_list[i]

        val = yr_bins.sel(year=yr).isel(bin_number=idx).values
        results.append(val)

    results = np.array(results)

    # -----------------------------
    # Step 3: compute anomalies vs climatology
    # -----------------------------
    anomaly = results - clim_05

    # -----------------------------
    # Step 4: pick extreme year
    # -----------------------------
    if extreme == "cold":
        worst_idx = np.argmin(anomaly)   # most negative deviation
    elif extreme == "hot":
        worst_idx = np.argmax(anomaly)   # most positive deviation
    else:
        raise ValueError(f"Extreme must be 'cold' or 'hot', Received {extreme}.")

    worst_year = years[worst_idx]

    return results, anomaly, worst_year

In [ ]:
def generate_candidate_months(
        cdf_monthly: xr.DataArray, 
        cdf_climatology: xr.DataArray,
        extreme: str = "cold",  # "cold" or "hot"
        ):

    """
    Run find_hot_cold_extreme_from_median() over entire input datasets.
    Generate a dataframe of selected years per month and simulation 
        for input to XMY generation function.

    Parameters
    ----------
    cdf_monthly : xr.DataArray
        Monthly CDF to compare against climatological CDF

    cdf_climatology : xr.DataArray
        CDF representing climatology

    extreme : str
        The type of shock XMY.
        "cold" -> pick minimum deviation
        "hot"  -> pick maximum deviation

    Returns
    -------
    top_df: dataframe of selected years per simulation and month
    """
    if extreme == "cold":
        var = "Daily min air temperature"
    elif extreme == "hot":
        var = "Daily max air temperature"
    
    subset_clim = cdf_climatology[var]
    subset_year = cdf_monthly[var]

    results = []
    for sim in cdf_monthly.sim.values:
        clim_sim  = subset_clim .sel(sim=sim)
        year_sim = subset_year.sel(sim=sim)
        for mon in cdf_monthly.month.values:
            clim_month  = clim_sim .sel(month=mon)
            year_month = year_sim.sel(month=mon)
            _, _, worst_year = find_hot_cold_extreme_from_median(year_month, clim_month, extreme=extreme) 
            row = {
            "month": mon,
            "sim": sim,
            "year": worst_year
        }
            results.append(row)
    top_df = pd.DataFrame(results)

    return top_df


In [ ]:
extreme = 'hot' # 'cold'
top_df = generate_candidate_months(cdf_monthly, cdf_climatology,extreme)

### Step 4: Generate the shock XMY data outputs

Generally, the following data is outputted using the XMY months:
- Date & time (UTC)
- Air temperature at 2m [°C]
- Dew point temperature [°C]
- Relative humidity [%]
- Global horizontal irradiance [W/m2]
- Direct normal irradiance [W/m2]
- Diffuse horizontal irradiance [W/m2]
- Downwelling infrared radiation [W/m2]
- Wind speed at 10m [m/s]
- Wind direction at 10m [°]
- Surface air pressure [Pa]

The following function will retrieve all of this data for the designated TMY month and concatenate it together into a pandas dataframe so that we can export it as a csv file. We'll do this next. 

In [ ]:
def generate_xmy_data(
    top_df: pd.DataFrame,
    start_year: int,
    end_year: int,
    lat: float,
    lon: float,
    data_models: list,
) -> dict:
    """Generate typical meteorological year data.

    Parameters
    ----------
    top_df : pd.DataFrame
        Table with columns month, simulation, and year. Each month-sim-yr
        combo represents the top candidate with the lowest weighted FS statistic.
    start_year : int
    end_year : int
    lat : float
    lon : float
    data_models : list

    Returns
    -------
    dict of str : pd.DataFrame
        Dictionary in the format {simulation: TMY DataFrame}.
    """
    # Define climate data object with minimal terminal output
    cd = ck.ClimateData(verbosity=-2)

    # Define variables and unit conversions
    var_info = {
        "t2": {
            "long_name": "Air temperature at 2m (degC)",
            "units": "degC",
        },  # convert default K -> C
        "dew_point_2m": {
            "long_name": "Dew point temperature at 2m (degC)",
            "units": "degC",
        },  # convert default K -> C
        "relative_humidity_2m": {
            "long_name": "Relative humidity (0-100)",
            "units": UNSET,
        },  # Use unit default (0-100)
        "swdnb": {
            "long_name": "Instantaneous downwelling shortwave flux at bottom (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "swddni": {
            "long_name": "Shortwave surface downward direct normal irradiance (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "swddif": {
            "long_name": "Shortwave surface downward diffuse irradiance (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "lwdnb": {
            "long_name": "Instantaneous downwelling longwave flux at bottom (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "wind_speed_10m": {
            "long_name": "Wind speed at 10m (m/s)",
            "units": UNSET,
        },  # Use unit default (m/s)
        "wind_direction_10m": {
            "long_name": "Wind direction at 10m (degrees)",
            "units": UNSET,
        },  # Use unit default (degrees)
        "psfc": {
            "long_name": "Surface pressure (Pa)",
            "units": UNSET,
        },  # Use unit default (Pa)
    }

    ## ================== GET DATA FROM CATALOG ==================
    print(
        f"STEP 1: RETRIEVING HOURLY DATA FROM CATALOG FOR {len(var_info)} VARIABLES\n"
    )
    all_vars_list = []
    for i, (var, info) in enumerate(var_info.items(), start=1):
        long_name = info["long_name"]
        units = info["units"]
        print(f"({i}/{len(var_info)}) Variable: {var}")
        print("Retrieving data from catalog...")
        data_by_var = (
            cd.catalog("cadcat")
            .variable(var)
            .activity_id("WRF")
            .institution_id("UCLA")
            .table_id("1hr")
            .grid_label("d03")
            .experiment_id(["historical", "ssp370"])
            .processes(
                {
                    "time_slice": (start_year, end_year + 1),
                    "clip": (lat, lon),
                    "convert_units": units,
                    "convert_to_local_time": {"convert": "yes"},
                }
            )
            .get()
        )

        # Subset simulations and convert to DataArray
        data_by_var = data_by_var.sel(
            sim=data_models, time=slice(str(start_year), str(end_year))
        )[var]

        # Update variable name to use long_name for clarity
        data_by_var.name = long_name

        print(f"Retrieved. Loading data into memory...")
        with ProgressBar():
            data_by_var.load()

        print(f"Data loaded into memory.\n")
        all_vars_list.append(data_by_var)

    # Merge data from all variables into a single xr.Dataset object
    # Drop unneeded coordinates
    all_vars_ds = xr.merge(all_vars_list)
    all_vars_ds = all_vars_ds.drop_vars(
        ["lakemask", "landmask", "x", "y", "Lambert_Conformal"], errors="ignore"
    )

    # Clear individual DataArrays from memory after merge
    del all_vars_list

    ## ================== CONSTRUCT TMY ==================
    print("\nSTEP 2: CALCULATING SHOCK EXTREME METEOROLOGICAL YEAR PER MODEL SIMULATION\n")
    tmy_df_all = {}
    for sim in all_vars_ds.sim.values:
        df_list = []
        print(f"Calculating TMY for simulation: {sim}")
        for mon in np.arange(1, 13, 1):
            # Get year corresponding to month and simulation combo
            year = top_df.loc[
                (top_df["month"] == mon) & (top_df["sim"] == sim)
            ].year.item()

            # Select data for unique month, year, and simulation
            data_at_stn_mon_sim_yr = all_vars_ds.sel(
                sim=sim,
                time=(all_vars_ds.time.dt.month == mon)
                & (all_vars_ds.time.dt.year == year),
            ).expand_dims("sim")

            # Reformat as dataframe
            df_by_mon_sim_yr = data_at_stn_mon_sim_yr.to_dataframe().reset_index()

            # Reformat time index to remove seconds
            df_by_mon_sim_yr["time"] = pd.to_datetime(
                df_by_mon_sim_yr["time"].values
            ).strftime("%Y-%m-%d %H:%M")

            df_list.append(df_by_mon_sim_yr)

        # Concatenate all DataFrames together
        tmy_df_by_sim = pd.concat(df_list)
        tmy_df_all[sim] = tmy_df_by_sim

        print("XMY PROCESS COMPLETE.")

    return tmy_df_all

In the next cell, we will run the `generate_xmy_data` function which will retrieve, subset, and format the data for each month according to the TMY months for all requested variables. We have included print statements so you can watch the progress for each variable in each month as it builds. 

<span style="color:#FF0000">**Note**: <span style="color:#000000"> This will take time! On the Analytics Engine JupyterHub, this takes approximately 22 minutes. Progress bars will indicate the status of generating the TMY data for each simulation. 

In [ ]:
xmy_data_to_export = generate_xmy_data(
    top_df, start_year, end_year, stn_lat, stn_lon, data_models
)

Let's observe what the XMY data looks like for one of the simulations:

In [ ]:
simulation = "wrf_ucla_miroc6_ssp370_r1i1p1f1"
xmy_data_to_export[simulation].head(5)

Next, we visualize the XMY data itself.

In [ ]:
# Make plot using pandas
ax = xmy_data_to_export[simulation].plot(
    x="time",
    y=[
        col
        for col in xmy_data_to_export[simulation]
        if col not in ["time", "sim", "lat", "lon"]
    ],
    subplots=True,
    figsize=(12, 12),
    legend=True,
    fontsize=10,
)

# Plot formatting
for a in ax.flatten():
    a.xaxis.label.set_fontsize(12)
    a.yaxis.label.set_fontsize(12)
    a.legend(loc="upper right", fontsize=14)
plt.suptitle(
    f"Typical Meteorological Year (simulation: {simulation})", fontsize=16, y=0.98
)
plt.tight_layout();

Lastly, let's export the XMY data below as csv files. There will be a file per simulation downloaded. When utilizing TMY data in your own workflows, we recommend that **all simulations are considered** in your analyses, especially for future scenarios. Note, if you are working with a custom location, please also provide the optional stn_elev argument to write_tmy_file; if no elevation value is provided, an elevation value of 0.0 is set as the default.

In [ ]:
import datetime
import os
import shutil
import urllib
import warnings
from importlib.metadata import version as _version
from math import prod
from typing import Tuple

import boto3
import botocore
import numpy as np
import pandas as pd
import pytz
import requests
import xarray as xr
from timezonefinder import TimezoneFinder
from dask.diagnostics import ProgressBar

from climakitae.core.paths import (
    EXPORT_S3_BUCKET,
    HADISD_STATIONS_URL,
    VARIABLE_DESCRIPTIONS_CSV_PATH,
)
from climakitae.util.utils import read_csv_file

xr.set_options(keep_attrs=True)
bytes_per_gigabyte = 1024 * 1024 * 1024
degree_sign = "\N{DEGREE SIGN}"


def remove_zarr(filename: str):
    """Remove Zarr directory structure helper function. As Zarr format is a directory
    tree it is not easily removed using JupyterHUB GUI. This function simply deletes
    an entire directory tree.

    Parameters
    ----------
    filename : str
        Output Zarr file name (without file extension, i.e. "my_filename" instead
        of "my_filename.zarr").

    """
    if type(filename) is not str:
        raise Exception(
            (
                "Please pass a string"
                " (any characters surrounded by quotation marks)"
                " for your file name."
            )
        )
    filename = filename.split(".")[0]

    dir_path = filename + ".zarr"

    try:
        shutil.rmtree(dir_path)
        print(f"Zarr dataset '{dir_path}' deleted successfully.")
    except FileNotFoundError:
        print(f"Zarr dataset '{dir_path}' not found.")
    except OSError as e:
        print(f"Error deleting Zarr dataset '{dir_path}': {e}")


def _add_metadata(data: xr.Dataset):
    """Add attributes to xarray dataset in-place.

    Parameters
    ----------
    data : xr.Dataset

    """
    ds_attrs = data.attrs

    ct = datetime.datetime.now()
    ct_str = ct.strftime("%d-%b-%Y (%H:%M)")

    ck_attrs = {
        "Data_exported_from": "Cal-Adapt Analytics Engine",
        "Data_export_timestamp": ct_str,
        "Analysis_package_name": "climakitae",
        "Version": _version("climakitae"),
        "Author": "Cal-Adapt Analytics Engine Team",
        "Author_email": "analytics@cal-adapt.org",
        "Home_page": "https://github.com/cal-adapt/climakitae",
        "License": "BSD 3-Clause License",
    }

    ds_attrs.update(ck_attrs)
    data.attrs = ds_attrs


def _estimate_file_size(data: xr.DataArray | xr.Dataset, format: str) -> float:
    """Estimate uncompressed file size in gigabytes when exporting `data` in `format`.

    Parameters
    ----------
    data : xr.DataArray | xr.Dataset
        data to export to the specified `format`
    format : str
        file format ("Zarr", "NetCDF", "CSV")

    Returns
    -------
    float
        estimated file size in gigabytes

    """
    match format:
        case "NetCDF" | "Zarr":
            data_size = data.nbytes
            buffer_size = 100 * 1024 * 1024  # 100 MB for miscellaneous metadata
            est_file_size = data_size + buffer_size
        case "CSV":
            # Rough estimate of the number of chars per CSV line
            # Will overestimate uncompressed size by 10-20%
            chars_per_line = 150

            match data:
                case xr.DataArray():
                    est_file_size = data.size * chars_per_line
                case xr.Dataset():
                    est_file_size = prod(data.sizes.values()) * chars_per_line
        case _:
            raise Exception('format needs to be "NetCDF", "Zarr", "CSV"')

    return est_file_size / bytes_per_gigabyte


def _warn_large_export(file_size: float, file_size_threshold: float | int = 5):
    """Print warning message if predicted file size exceeds threshold.

    Parameters
    ----------
    file_size : float
        Predicted file size in GB.
    file_size_threshold : float | int
        Threshold size in GB for warning.

    """
    if file_size > file_size_threshold:
        print(
            "WARNING: Estimated file size is "
            + str(round(file_size, 2))
            + " GB. This might take a while!"
        )


def _update_encoding(data: xr.Dataset):
    """Update data encodings to prevent issues when exporting them to NetCDF.

    Drop `missing_value` encoding, if any, on `data` as well as its coordinates
    and data variables.

    Parameters
    ----------
    data : xr.Dataset

    Returns
    -------
    None

    Notes
    -----
    These encoding updates resolve errors raised when writing NetCDF files to
    S3.

    """

    def _unencode_missing_value(d: xr.Dataset):
        """Drop `missing_value` encoding, if any, on data object `d`.

        Parameters
        ----------
        d : xr.Dataset

        Returns
        -------
        None

        """
        for key in [
            "missing_value",
            "compressor",
            "filters",
            "chunks",
            "preferred_chunks",
        ]:
            try:
                del d.encoding[key]
            except:
                pass

    _unencode_missing_value(data)
    for coord in data.coords:
        _unencode_missing_value(data[coord])

    for data_var in data.data_vars:
        _unencode_missing_value(data[data_var])


def _fillvalue_encoding(data: xr.Dataset) -> dict[str, int | float | None]:
    """Creates FillValue encoding for each variable for export to NetCDF.

    Parameters
    ----------
    data : xr.Dataset

    Returns
    -------
    encoding : dict

    """
    fill = dict(_FillValue=None)
    filldict = {coord: fill for coord in data.coords}
    return filldict


def _compression_encoding(data: xr.Dataset) -> dict[str, int | float | None]:
    """Creates compression encoding for each variable for export to NetCDF.

    Parameters
    ----------
    data : xr.Dataset

    Returns
    -------
    encoding : dict

    """
    comp = dict(zlib=True, complevel=6)
    compdict = {var: comp for var in data.data_vars}
    return compdict


def _convert_da_to_ds(data: xr.DataArray | xr.Dataset) -> xr.Dataset:
    """Convert xarray data array to dataset.

    Parameters
    ----------
    data : xr.DataArray | xr.Dataset

    """
    match data:
        case xr.DataArray():
            if not data.name:
                # name it in order to call to_dataset on it
                data.name = "data"
            return data.to_dataset()
        case xr.Dataset():
            return data
        case _:
            raise Exception("Input must be either an Xarray DataArray or Dataset")


def _export_to_netcdf(data: xr.DataArray | xr.Dataset, save_name: str):
    """Export user-selected data to NetCDF format.

    Export the xarray DataArray or Dataset `data` to a NetCDF file `save_name`.
    If there is enough disk space, the function saves the file locally to the
    jupyter hub.


    Parameters
    ----------
    data : xr.DataArray | xr.Dataset
        data to export to NetCDF format
    save_name : str
        desired output file name, including the file extension

    Returns
    -------
    None

    """
    print("Exporting specified data to NetCDF...")

    # Convert xr.DataArray to xr.Dataset so that compression can be utilized
    _data = _convert_da_to_ds(data)

    est_file_size = _estimate_file_size(_data, "NetCDF")
    disk_space = shutil.disk_usage(os.path.expanduser("~"))[2] / bytes_per_gigabyte

    _warn_large_export(est_file_size)

    _add_metadata(_data)

    _update_encoding(_data)

    if disk_space <= est_file_size:
        raise Exception(
            "Data too large to save locally. Use the format='Zarr', mode='s3' options."
        )

    print("Saving file locally as NetCDF4...")
    path = os.path.join(os.getcwd(), save_name)

    if os.path.exists(path):
        raise Exception(
            (
                f"File {save_name} exists. "
                "Please either delete that file from the work space "
                "or specify a new file name here."
            )
        )
    encoding = _fillvalue_encoding(_data) | _compression_encoding(_data)

    # Recursively validate attribute types
    def validate_attrs(obj):
        """Recursively validate that all attributes are of allowed types."""
        allowed_types = (
            str,
            int,
            float,
            complex,
            np.ndarray,
            list,
            tuple,
            bytes,
            np.integer,
            np.floating,
            np.complexfloating,
        )

        if hasattr(obj, "attrs"):
            for key, value in list(obj.attrs.items()):
                # Explicitly handle boolean values which are subclasses of int but cause issues in NetCDF
                if isinstance(value, (bool, np.bool_)):
                    obj.attrs[key] = str(value)
                    continue

                if value is not None and not isinstance(value, allowed_types):
                    # Convert or remove problematic attributes
                    try:
                        obj.attrs[key] = str(value)
                    except:
                        del obj.attrs[key]

    # Validate dataset attributes
    validate_attrs(_data)

    # Validate coordinate attributes
    for coord in _data.coords:
        validate_attrs(_data[coord])

    # Validate data variable attributes
    for var in _data.data_vars:
        validate_attrs(_data[var])

    # Unify chunks
    _data = _data.unify_chunks()

    # Export to NetCDF with ProgressBar
    delayed_write = _data.to_netcdf(
        path, format="NETCDF4", engine="netcdf4", encoding=encoding, compute=False
    )
    with ProgressBar():
        delayed_write.compute()

    print(
        (
            "Saved! You can find your file in the panel to the left"
            " and download to your local machine from there."
        )
    )


def _export_to_zarr(data: xr.DataArray | xr.Dataset, save_name: str, mode: str):
    """Export user-selected data to Zarr format.
    Export the xarray DataArray or Dataset `data` to a Zarr dataset `save_name`.
    If `local` mode used it is saved to the HUB user partition. If `s3` mode used
    it is saved to the AWS S3 bucket `cadcat-tmp` and provides a URL for download.

    Parameters
    ----------
    data : xr.DataArray | xr.Dataset
        data to export to Zarr format
    save_name : str
        desired output Zarr directory name
    mode : str
        location logic for storing export file (`local`, `s3`)

    Returns
    -------
    None

    """
    print("Exporting specified data to Zarr...")

    # Convert xr.DataArray to xr.Dataset so that compression can be utilized
    _data = _convert_da_to_ds(data)

    # Unify chunks to avoid errors with inconsistent chunking
    _data = _data.unify_chunks()

    est_file_size = _estimate_file_size(_data, "Zarr")
    disk_space = shutil.disk_usage(os.path.expanduser("~"))[2] / bytes_per_gigabyte

    _warn_large_export(est_file_size)

    _add_metadata(_data)

    _update_encoding(_data)

    def _write_zarr(path: str, data: xr.Dataset):
        # Ensure encoding is clean
        for var in data.variables:
            # Clear all encoding that might cause issues
            data[var].encoding.clear()

        encoding = _fillvalue_encoding(data)
        chunks = {k: v[0] for k, v in data.chunks.items()}
        data = data.chunk(chunks)
        with ProgressBar():
            data.to_zarr(path, encoding=encoding)

    def _write_zarr_to_s3(
        display_path: str, path: str, save_name: str, data: xr.Dataset
    ):
        _write_zarr(path, data)

        print(
            (
                "Saved! To open the file in your local machine, "
                "open the following S3 URI using xarray:"
                "\n\n"
                f"{display_path}"
                "\n\n"
                "Example of opening and saving to netCDF:\n"
                "ds = xr.open_zarr('"
                + display_path
                + "', storage_options={'anon': True})\n"
                "comp = dict(zlib=True, complevel=6)\n"
                "compdict = {var: comp for var in ds.data_vars}\n"
                "ds.to_netcdf('"
                + save_name.rstrip(".zarr")
                + ".nc', encoding=compdict)\n"
                "\n\n"
                ""
                "Note: The URL will remain valid for 1 week."
            )
        )

    match mode:
        case "local":
            print("Saving file locally as Zarr...")
            if disk_space <= est_file_size:
                raise Exception(
                    "Data too large to save locally. Use the format='Zarr', mode='s3' options."
                )
            path = os.path.join(os.getcwd(), save_name)

            if os.path.exists(path):
                raise Exception(
                    (
                        f"File {save_name} exists. "
                        "Please either delete that file from the work space "
                        "or specify a new file name."
                    )
                )
            _write_zarr(path, _data)
        case "s3":
            print("Saving file to S3 scratch bucket as Zarr...")
            display_path = f"{os.environ['SCRATCH_BUCKET']}/{save_name}"
            path = "simplecache::" + display_path
            prefix = display_path.split(EXPORT_S3_BUCKET + "/")[-1]

            s3 = boto3.resource("s3")
            try:
                s3.Object(EXPORT_S3_BUCKET, prefix + "/.zattrs").load()
            except botocore.exceptions.ClientError as e:
                if e.response["Error"]["Code"] == "404":
                    # The object does not exist so go ahead and write to S3
                    _write_zarr_to_s3(display_path, path, save_name, _data)
                else:
                    # Something else has gone wrong.
                    raise
            else:
                # The object does exist
                raise Exception(f"File {save_name} exists. Specify a new file name.")
        case _:
            raise Exception("Correct mode not specified. Use either 'local' or 's3'.")


def _get_unit(dataarray: xr.DataArray) -> str:
    """Return unit of data variable in `dataarray`, if any, or an empty string.

    Parameters
    ----------
    dataarray : xr.DataArray

    Returns
    -------
    str

    """
    data_attrs = dataarray.attrs
    if "units" in data_attrs and data_attrs["units"] is not None:
        return data_attrs["units"]
    else:
        return ""


def _ease_access_in_R(column_name: str) -> str:
    """Return a copy of the input that can be used in R easily.

    Modify the `column_name` string so that when it is the name of an R data
    table column, the column can be accessed by $. The modified string contains
    no spaces or special characters, and starts with a letter or a dot.

    Parameters
    ----------
    column_name : str

    Returns
    -------
    str

    Notes
    -----
    The input is assumed to be a column name of a pandas DataFrame converted
    from an xarray DataArray or Dataset available on the Cal-Adapt Analytics
    Engine. The conversions are through the to_dataframe method.

    The function acts on one of the display names of the variables:
    https://github.com/cal-adapt/climakitae/blob/main/climakitae/data/variable_descriptions.csv
    or one of the station names:
    s3://cadcat/hadisd/hadisd_stations.csv

    """
    return (
        column_name.replace("(", "")
        .replace(")", "")
        .replace(" ", "_")
        .replace("-", "_")
    )


def _update_header(
    df: pd.DataFrame, variable_unit_map: list[tuple[str, str]]
) -> pd.DataFrame:
    """Update data table header to match the given variable names and units.

    Update the header of the DataFrame `df` so that name and unit of the data
    variable contained in each column are as specified in `variable_unit_map`.
    The resulting header starts with a row labeled "variable" holding variable
    names. A 2nd "unit" row include the units associated with the variables.


    Parameters
    ----------
    df : pd.DataFrame
        data table to update
    variable_unit_map : list[tuple[str, str]]
        list of tuples where each tuple contains the name and unit of the data
        variable in a column of the input data table

    Returns
    -------
    pd.DataFrame
        data table with updated header

    """
    df.columns = pd.MultiIndex.from_tuples(
        variable_unit_map,
        name=["variable", "unit"],
    )
    df.reset_index(inplace=True)  # simplifies header
    return df


def _dataarray_to_dataframe(dataarray: xr.DataArray) -> pd.DataFrame:
    """
    Prepare xarray DataArray for export as CSV file.

    Convert the xarray DataArray `dataarray` to a pandas DataFrame ready to be
    exported to CSV format. The DataArray is converted through its to_dataframe
    method. The DataFrame header is renamed as needed to ease the access of
    columns in R. It is also enriched with the unit associated with the data
    variable in the DataArray.

    Parameters
    ----------
    dataarray : xr.DataArray
        data to be prepared for export

    Returns
    -------
    pd.DataFrame
        data ready for export

    """
    if not dataarray.name:
        # name it in order to call to_dataframe on it
        dataarray.name = "data"

    df = dataarray.to_dataframe()

    variable = dataarray.name
    unit = _get_unit(dataarray)
    variable_unit_map = []
    for col in df.columns:
        if col == variable:
            variable_unit_map.append((_ease_access_in_R(col), unit))
        else:
            variable_unit_map.append((_ease_access_in_R(col), ""))

    df = _update_header(df, variable_unit_map)
    return df


def _dataset_to_dataframe(dataset: xr.Dataset) -> pd.DataFrame:
    """Prepare xarray Dataset for export as CSV file.

    Convert the xarray Dataset `dataset` to a pandas DataFrame ready to be
    exported to CSV format. The Dataset is converted through its to_dataframe
    method. The DataFrame header is renamed as needed to ease the access of
    columns in R. It is also enriched with the units associated with the data
    variables and other non-index variables in the Dataset. If the Dataset
    contains station data, the name of any climate variable associated with
    the station(s) is added to the header as well.

    Parameters
    ----------
    dataset : xr.Dataset
        data to be prepared for export

    Returns
    -------
    pd.DataFrame
        data ready for export

    """
    df = dataset.to_dataframe()

    variable_unit_map = [
        (var_name, _get_unit(dataset[var_name])) for var_name in df.columns
    ]
    df = _update_header(df, variable_unit_map)

    # Helpers for adding to header climate variable names associated w/ stations
    station_df = pd.read_csv(HADISD_STATIONS_URL)
    station_lst = list(station_df.station)

    def _is_station(name):
        """Return True if `name` is an HadISD station name."""
        return name in station_lst

    variable_description_df = read_csv_file(VARIABLE_DESCRIPTIONS_CSV_PATH)
    variable_ids = variable_description_df.variable_id.values

    def _variable_id_to_name(var_id: str) -> str:
        """Convert variable ID to variable name.

        Return the "display_name" associated with the "variable_id" in
        variable_descriptions.csv. If `var_id` is not a "variable_id" in the
        CSV file, return an empty string.

        Parameters
        ----------
        var_id : str

        Returns
        -------
        str

        """
        if var_id in variable_ids:
            var_name_series = variable_description_df.loc[
                variable_ids == var_id, "display_name"
            ]
            var_name = var_name_series.to_list()[0]
            return var_name
        else:
            return ""

    def _get_station_variable_name(dataset: xr.Dataset, station: str) -> str:
        """Get name of climate variable stored in `dataset` variable `station`.

        Return an empty string if that is not possible.

        Parameters
        ----------
        dataset : xr.Dataset
        station : str

        Returns
        -------
        var_name : str

        """
        try:
            station_da = dataset[station]  # DataArray
            data_attrs = station_da.attrs
            if "variable_id" in data_attrs and data_attrs["variable_id"] is not None:
                var_id = data_attrs["variable_id"]
                var_name = _variable_id_to_name(var_id)
                return var_name
            else:
                return ""
        except:
            return ""

    # Add to header: name of any climate variable associated with stations
    column_names = df.columns.get_level_values(0)
    climate_var_lst = []
    for name in column_names:
        if _is_station(name):
            climate_var = _get_station_variable_name(dataset, station=name)
        else:
            climate_var = ""
        climate_var_lst.append(climate_var)

    if set(climate_var_lst) != {""}:
        # Insert climate variable names to the 2nd row
        header_df = df.columns.to_frame()
        header_df.insert(1, "", climate_var_lst)
        # The 1st row was named "variable" by `_update_header`
        header_df.variable = header_df.variable.map(_ease_access_in_R)
        df.columns = pd.MultiIndex.from_frame(header_df)

    return df


def _export_to_csv(data: xr.DataArray | xr.Dataset, save_name: str):
    """Export user-selected data to CSV format.

    Export the xarray DataArray or Dataset `data` to a CSV file at
    `output_path`.

    Parameters
    ----------
    data : xr.DataArray | xr.Dataset
        data to export to CSV format
    save_name : str
        desired export file prefix

    Returns
    -------
    None

    """
    # Check file size and avail workspace disk space
    # raise error for not enough space
    # and warning for large file
    est_file_size = _estimate_file_size(data, "CSV")
    disk_space = shutil.disk_usage(os.path.expanduser("~"))[2] / bytes_per_gigabyte

    if disk_space <= est_file_size:
        raise Exception(
            "Not enough disk space to export data! You need at least "
            + str(round(est_file_size, 2))
            + (
                " GB free in the hub directory, which has 10 GB total space."
                " Try smaller subsets of space, time, scenario, and/or"
                " simulation; pick a coarser spatial or temporal scale;"
                " or delete any exported datasets which you have already"
                " downloaded or do not want."
            )
        )

    # Check if export file already exists and exit if so
    output_path = os.path.join(os.getcwd(), save_name)
    if os.path.exists(output_path):
        raise Exception(
            (
                f"File {save_name} exists. "
                "Please either delete that file from the work space "
                "or specify a new file name here."
            )
        )

    print("Exporting specified data to CSV...")
    _warn_large_export(est_file_size, 1.0)

    match data:
        case xr.DataArray():
            df = _dataarray_to_dataframe(data)
        case xr.Dataset():
            df = _dataset_to_dataframe(data)
        case _:
            raise Exception("Input data needs to be an Xarray DataArray or Dataset")

    # Warn about exceedance of Excel row or column limit
    excel_row_limit = 1048576
    excel_column_limit = 16384
    csv_nrows, csv_ncolumns = df.shape
    if csv_nrows > excel_row_limit or csv_ncolumns > excel_column_limit:
        warnings.warn(
            f"Dataset exceeds Excel limits of {excel_row_limit} rows "
            f"and {excel_column_limit} columns.",
            stacklevel=999,
        )

    def _metadata_to_file(ds: xr.Dataset, output_name: str):
        """Write NetCDF metadata to a txt file so users can still access it
        after exporting to a CSV.

        Parameters
        ----------
        ds : xr.Dataset
        output_name : str

        Returns
        -------
        None

        """

        def _rchop(s, suffix):
            if suffix and s.endswith(suffix):
                return s[: -len(suffix)]
            return s

        output_name = _rchop(output_name, ".csv.gz")

        if os.path.exists(output_name + "_metadata.txt"):
            os.remove(output_name + "_metadata.txt")

        print(
            "NOTE: File metadata will be written in "
            + output_name
            + (
                "_metadata.txt. We recommend you download this along with "
                "the CSV for your records."
            )
        )

        with open(output_name + "_metadata.txt", "w") as f:
            f.write("======== Metadata for CSV file " + output_name + " ========")
            f.write("\n")
            f.write("\n")
            f.write("\n")
            f.write("===== Global file attributes =====")
            if type(ds) == xr.core.dataarray.DataArray:
                f.write("\n")
                f.write("Name: " + ds.name)
            f.write("\n")
            for att_keys, att_values in ds.attrs.items():
                f.write(str(att_keys) + " : " + str(att_values))
                f.write("\n")

            f.write("\n")
            f.write("\n")
            f.write("===== Coordinate descriptions =====")
            f.write("\n")
            f.write("Note: coordinate values are in the CSV")
            f.write("\n")

            for coord in ds.coords:
                f.write("\n")
                f.write("== " + str(coord) + " ==")
                f.write("\n")
                for att_keys, att_values in ds[coord].attrs.items():
                    f.write(str(att_keys) + " : " + str(att_values))
                    f.write("\n")

            if type(ds) == xr.core.dataset.Dataset:
                f.write("\n")
                f.write("\n")
                f.write("===== Variable descriptions =====")
                f.write("\n")

                for var in ds.data_vars:
                    f.write("\n")
                    f.write("== " + str(var) + " ==")
                    f.write("\n")
                    for att_keys, att_values in ds[var].attrs.items():
                        f.write(str(att_keys) + " : " + str(att_values))
                        f.write("\n")

    _metadata_to_file(data, output_path)
    df.to_csv(output_path, compression="gzip")
    print(
        (
            "Saved! You can find your file(s) in the panel to the left"
            " and download to your local machine from there."
        )
    )


def export(
    data: xr.DataArray | xr.Dataset,
    filename: str = "dataexport",
    format: str = "NetCDF",
    mode: str = "local",
):
    """Save xarray data as NetCDF, Zarr, or CSV in the current working directory, or if Zarr optionally
    stream the export file to an AWS S3 scratch bucket and give download URL. NetCDF can only be written
    to the HUB user partition if it will fit. Zarr can either be written to the HUB user partition or to
    S3 scratch bucket using the mode option.

    Parameters
    ----------
    data : xr.DataArray | xr.Dataset
        Data to export, as output by e.g. `DataParameters.retrieve()`.
    filename : str, optional
        Output file name (without file extension, i.e. "my_filename" instead
        of "my_filename.nc"). The default is "dataexport".
    format : str, optional
        File format ("Zarr", "NetCDF", "CSV"). The default is "NetCDF".
    mode : str, optional
        Save location logic for Zarr file ("local", "s3"). The default is "local"

    Returns
    -------
    None

    """
    ftype = type(data)

    if ftype not in [xr.core.dataset.Dataset, xr.core.dataarray.DataArray]:
        raise Exception(
            "Cannot export object of type "
            + str(ftype).strip("<class >")
            + ". Please pass an Xarray Dataset or DataArray."
        )

    if type(filename) is not str:
        raise Exception(
            (
                "Please pass a string"
                " (any characters surrounded by quotation marks)"
                " for your file name."
            )
        )
    filename = filename.split(".")[0]

    req_format = format.lower()

    if req_format not in ["zarr", "netcdf", "csv"]:
        raise Exception('Please select "Zarr", "NetCDF" or "CSV" as the file format.')

    extension_dict = {"zarr": ".zarr", "netcdf": ".nc", "csv": ".csv.gz"}

    save_name = filename + extension_dict[req_format]

    if (mode == "s3") and (req_format != "zarr"):
        raise Exception('To export to AWS S3 you must use the format="Zarr" option.')

    # now here is where exporting actually begins
    # we will have different functions for each file type
    # to keep things clean-ish
    match req_format:
        case "zarr":
            _export_to_zarr(data, save_name, mode)
        case "netcdf":
            _export_to_netcdf(data, save_name)
        case "csv":
            _export_to_csv(data, save_name)
        case _:
            raise Exception(
                'Please select "Zarr", "NetCDF" or "CSV" as the file format.'
            )


## TMY export functions
def _grab_dem_elev_m(lat: float, lon: float) -> float:
    """Pulls elevation value from the USGS Elevation Point Query Service,
    lat lon must be in decimal degrees (which it is after cleaning)
    Modified from:
    https://gis.stackexchange.com/questions/338392/getting-elevation-for-multiple-lat-long-coordinates-in-python

    Note: This is breaking at present (2/29/2024) -- setting to pulling station elevation from csv, 0 for custom

    Parameters
    ----------
    lat : float
        latitude of point of interest
    lon : float
        longitude of point of interest

    Returns
    -------
    float
        elevation at point of interest

    """
    url = r"https://epqs.nationalmap.gov/v1/json?"

    # define rest query params
    params = {"output": "json", "x": lon, "y": lat, "units": "Meters"}

    # format query string and return value
    result = requests.get((url + urllib.parse.urlencode(params)))

    # error checking on api call
    if "value" not in result.json():
        print("Please re-run the current cell to re-try the API call")
    else:
        dem_elev_long = float(result.json()["value"])
        # make sure to round off lat-lon values so they are not improbably precise for our needs
        dem_elev_short = np.round(dem_elev_long, decimals=2)

    return dem_elev_short.astype("float")


def _epw_format_data(df: pd.DataFrame) -> pd.DataFrame:
    """Constructs TMY output file in specific order and missing data codes
    Source: EnergyPlus Version 23.1.0 Documentation

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    df : pd.DataFrame

    """
    # Normalize internal variable names to EPW column names.
    # The TMY pipeline uses short display names (e.g. "Air Temperature at 2m")
    # but EPW format expects names with units (e.g. "Air temperature at 2m (degC)").
    _internal_to_epw = {
        "Air Temperature at 2m": "Air temperature at 2m (degC)",
        "Dew point temperature": "Dew point temperature at 2m (degC)",
        "Relative humidity": "Relative humidity (0-100)",
        "Surface Pressure": "Surface pressure (Pa)",
        "Instantaneous downwelling shortwave flux at bottom": "Instantaneous downwelling shortwave flux at bottom (W/m2)",
        "Shortwave surface downward direct normal irradiance": "Shortwave surface downward direct normal irradiance (W/m2)",
        "Shortwave surface downward diffuse irradiance": "Shortwave surface downward diffuse irradiance (W/m2)",
        "Wind direction at 10m": "Wind direction at 10m (degrees)",
        "Wind speed at 10m": "Wind speed at 10m (m/s)",
        "Instantaneous downwelling longwave flux at bottom": "Instantaneous downwelling longwave flux at bottom (W/m2)",
    }
    df = df.rename(columns=_internal_to_epw)

    # set time col to datetime object for easy split
    df["time"] = pd.to_datetime(df["time"])
    if "warming_level" in df.columns:
        # change year for GWL data to not use 2000's dummy times
        df = df.assign(
            year=df["time"].dt.year - 2000,
            month=df["time"].dt.month,
            day=df["time"].dt.day,
            hour=df["time"].dt.hour + 1,  # 1-24, not 0-23
            minute=df["time"].dt.minute,
        )
    else:
        df = df.assign(
            year=df["time"].dt.year,
            month=df["time"].dt.month,
            day=df["time"].dt.day,
            hour=df["time"].dt.hour + 1,  # 1-24, not 0-23
            minute=df["time"].dt.minute,
        )

    # set epw variable order, very specific -- manually set
    # Note: vars not provided by AE are noted as missing
    colnames = [
        "year",
        "month",
        "day",
        "hour",
        "minute",
        "data_source",  # missing
        "Air temperature at 2m (degC)",
        "Dew point temperature at 2m (degC)",
        "Relative humidity (0-100)",
        "Surface pressure (Pa)",
        "exthorrad",  # missing - extraterrestrial horizontal radiation
        "extdirrad",  # missing - extraterrestrial direct normal radiation
        "Instantaneous downwelling longwave flux at bottom (W/m2)",
        "Instantaneous downwelling shortwave flux at bottom (W/m2)",
        "Shortwave surface downward direct normal irradiance (W/m2)",
        "Shortwave surface downward diffuse irradiance (W/m2)",
        "glohorillum",  # missing - global horizontal illuminance (lx)
        "dirnorillum",  # missing - direct normal illuminance (lx)
        "difhorillum",  # missing - diffuse horizontal illuminance (lx)
        "zenlum",  # missing - zenith luminnace (lx)
        "Wind direction at 10m (degrees)",
        "Wind speed at 10m (m/s)",
        "totskycvr",  # missing - total sky cover (tenths)
        "opaqskycvr",  # missing - opaque sky cover (tenths)
        "visibility",  # missing - visibility (km)
        "ceiling_hgt",  # missing - ceiling height (m)
        "presweathobs",  # missing - present weather observation
        "presweathcodes",  # missing - present weather codes
        "precip_wtr",  # missing - precipitatble water (mm)
        "aerosol_opt_depth",  # missing - aerosol optical depth (thousandths)
        "snowdepth",  # missing - snow depth (cm)
        "days_last_snow",  # missing - days since last snow
        "albedo",  # missing - albedo
        "liq_precip_depth",  # missing - liquid precip depth (mm)
        "liq_precip_rate",  # missing - liquid precip rate (h)
    ]

    # set specific missing data flags per variable
    for var in [
        "exthorrad",
        "extdirrad",
        "dirnorrad",
        "zenlum",
        "visibility",
    ]:
        df[var] = 9999
    for var in ["glohorillum", "dirnorillum", "difhorillum"]:
        df[var] = 999900
    for var in ["days_last_snow", "liq_precip_rate"]:
        df[var] = 99
    for var in ["precip_wtr", "snowdepth", "albedo", "liq_precip_depth"]:
        df[var] = 999
    df["ceiling_hgt"] = 99999
    df["aerosol_opt_depth"] = 0.999
    df["presweathobs"] = 9
    df["presweathcodes"] = 999999999

    # Note: Setting cloud cover to 5, per stakeholder recommendation, indicates 5/10ths skycover = 50% cloudy
    # Note: At present AE has no plans to provide cloud coverage data
    for var in ["totskycvr", "opaqskycvr"]:
        df[var] = 5

    # lastly set data source / uncertainty flag (section 2.13 of doc)
    # on AE: ? = var does not fit source options
    # on AE: 9 = uncertainty unknown
    df["data_source"] = "?9?9?9?9?9?9?9?9?9?9?9?9?9?9?9?9?9?9?9?9?9?9?9?9?9"

    # resets col order and drops any unnamed column from original df
    df = df.reindex(columns=colnames)

    return df


def _leap_day_fix(df: pd.DataFrame) -> pd.DataFrame:
    """Addresses leap day inclusion in TMY dataframe bug by removing the extra nan rows and resetting time index to Feb 28

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    df_leap : pd.DataFrame

    """
    df_leap = df.copy(deep=True)
    df_leap["time"] = pd.to_datetime(df["time"])  # set time to datetime
    df_leap = df_leap.dropna()  # drops extra rows

    # 3 models have leap days, 1 model does not -- handling for both
    # handling for TaiESM1 (no leap day natively)
    match df_leap.sim.unique()[0]:
        case "wrf_ucla_taiesm1_ssp370_r1i1p1f1":
            df_leap["time"] = np.where(
                (df_leap.time.dt.month == 2) & (df_leap.time.dt.day == 29),
                df_leap.time - pd.DateOffset(days=1),
                df_leap.time,
            )  # reset remaining feb 29 hours to feb 28

        # handling for 3 models with native leap days
        case _:
            df_leap["time"] = pd.to_datetime(df["time"])  # set time to datetime
            df_leap = df_leap.loc[
                ~((df_leap.time.dt.month == 2) & (df_leap.time.dt.day == 29))
            ]

    return df_leap


def _find_missing_val_month(df: pd.DataFrame) -> int:
    """Finds month that does not match expected hours

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    int

    """
    hrs_per_month = {
        1: 744,
        2: 672,
        3: 744,
        4: 720,
        5: 744,
        6: 720,
        7: 744,
        8: 744,
        9: 720,
        10: 744,
        11: 720,
        12: 744,
    }
    for m in range(1, 13, 1):
        df_month = df.loc[df.time.dt.month == m]
        if len(df_month) != hrs_per_month[m]:
            return m


def _missing_hour_fix(df: pd.DataFrame) -> pd.DataFrame:
    """Addresses missing hour in TMY dataframe bug by adding the missing hour at the appropriate spot and duplicating the previous hour's values

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    df_fixed : pd.DataFrame

    Notes
    -----
    Only fixes missing hour if missing hour is not the first or last hour of the month.

    """
    df_missing = df.copy(deep=True)
    df_missing["time"] = pd.to_datetime(df["time"])  # set time to datetime

    # first identify where missing hour is
    miss_month = _find_missing_val_month(
        df_missing
    )  # typically march or april when DST "goes forward an hour"

    # brute force way - as it is not a continuous time index (months are spliced together)
    df_prior = df_missing.loc[
        df_missing.time.dt.month < miss_month
    ]  # data prior to miss_month
    df_post = df_missing.loc[
        df_missing.time.dt.month > miss_month
    ]  # data after miss_month

    # fix missing hour
    df_bad = df_missing.loc[
        df_missing.time.dt.month == miss_month
    ]  # pulls out just the month with the missing hour
    df_bad.index = pd.to_datetime(df_bad.time)

    # set up correct df of month with all hours
    df_full = pd.DataFrame(
        pd.date_range(start=df_bad.index.min(), end=df_bad.index.max(), freq="h")
    )
    missing_cols = [col for col in df_bad.columns]
    df_full[missing_cols] = np.nan
    df_full["time"] = df_full[0]
    df_full.index = pd.to_datetime(df_full.time)

    df_month_fixed = pd.concat([df_bad, df_full.dropna(axis=1, how="all")])
    df_month_fixed = df_month_fixed.drop_duplicates(subset=["time"], keep="first")
    df_month_fixed = df_month_fixed.drop(columns=["time", 0])
    df_month_fixed = df_month_fixed.sort_values(by="time", ascending=True)
    df_month_fixed = df_month_fixed.reset_index()
    df_month_fixed = df_month_fixed.ffill()  # fill from previous days values

    # concat dfs together
    df_fixed = pd.concat([df_prior, df_month_fixed, df_post])
    return df_fixed


def _tmy_8760_size_check(df: pd.DataFrame) -> pd.DataFrame:
    """Checks the size of the TMY dataframe for export to ensure that it is explicitly 8760 in size.
    There are several scenarios where the input TMY dataframe would not be 8760 in size:
    (1) Size 8761, additional single hour due to time change for local time. Fix removes the duplicate row (typically in Nov.)
    (2) Size 8759, missing a single hour due to time change for local time. Fix adds the missing row (typically in Mar/Apr) by filling in from the previous hour.
    (3) Size 8784, 24 extra hours due to inclusion of a leap year February and specific models that retain leap days. Fix removes the additional rows.
    (4) Size 8783, 24 extra hours due to inclusion of a leap year February and a missing hour due to time change. Fix adds missing row and removes additional leap day rows.
    (5) Size 8758, missing two single hours due to time change for local time. Fix adds the missing row by filling in from the previous hour. Run twice.
        e.g. March 2008 and April 2000 are both time change months. Source: https://en.wikipedia.org/wiki/History_of_time_in_the_United_States

    Note: This is a bug introduced by the time zone correction to local time and should be addressed in the future.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe of TMY to export

    Returns
    -------
    df : pd.Dataframe
        Dataframe of TMY to export, explicitly 8760 in size

    """
    # first drop any duplicate time rows -- some df with 8760 are 8759 with duplicate rows, i.e., not a true 8760
    df_to_check = df.copy(deep=True)
    df_to_check = df_to_check.drop_duplicates(subset=["time"], keep="first")

    # fix cases
    match len(df_to_check):
        case 8760:
            return df_to_check
        case 8759:  # Missing hour, add missing row
            df_to_check = _missing_hour_fix(df_to_check)
            return df_to_check
        case 8784:  # Leap day added, remove Feb 29
            df_to_check = _leap_day_fix(df_to_check)
            return df_to_check
        case 8783:  # Leap day added and missing hour
            # remove leap day
            df_to_check = _leap_day_fix(df_to_check)
            # add missing hour
            df_to_check = _missing_hour_fix(df_to_check)
            return df_to_check
        case 8758:  # double missing hour
            df_to_check = _missing_hour_fix(df_to_check)  # march fix
            df_to_check = _missing_hour_fix(df_to_check)  # april fix
            return df_to_check
        case 8782:  # Leap day and double missing hour
            # remove leap day
            df_to_check = _leap_day_fix(df_to_check)
            # add missing hours
            df_to_check = _missing_hour_fix(df_to_check)  # march fix
            df_to_check = _missing_hour_fix(df_to_check)  # april fix
            return df_to_check
        case _:  # none of the above
            print(
                "Error: The size of the input dataframe ({}) does not comform to standard 8760 size. Please confirm.".format(
                    len(df)
                )
            )
            return None


def _tmy_reset_time_for_gwl(df: pd.DataFrame) -> pd.DataFrame:
    """
    Change dummy time in GWL data to start at year 0001
    for writing TMY results.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe of TMY data to export
    """

    def replace_year(datestr: str) -> str:
        """Subtract 2000 from dummy year to reset to 0001 baseline."""
        year = int(datestr.split("-")[0])
        year = year - 2000
        datestr = str(year).zfill(4) + "-" + "-".join(datestr.split("-")[1:])
        return datestr

    cleaned_years = [replace_year(str(t)) for t in df["time"]]
    df["time"] = cleaned_years
    return df


def write_tmy_file(
    filename_to_export: str,
    df: pd.DataFrame,
    years: Tuple[int, int],
    location_name: str,
    station_code: int,
    stn_lat: float,
    stn_lon: float,
    stn_state: str,
    stn_elev: float = 0.0,
    file_ext: str = "tmy",
):
    """Exports TMY data either as .epw or .tmy file

    Parameters
    ----------
    filename_to_export : str
        Filename string, constructed with station name and simulation
    df : pd.DataFrame
        Dataframe of TMY data to export
    years : Tuple[int, int]
        Tuple containing climatology start and end years
    location_name : str
        Location name string, often station name
    station_code : int
        Station code
    stn_lat : float
        Station latitude
    stn_lon : float
        Station longitude
    stn_state : str
        State of station location
    stn_elev : float, optional
        Elevation of station, default is 0.0
    file_ext : str, optional
        File extension for export, default is .tmy, options are "tmy" and "epw"

    Returns
    -------
    None

    """
    station_df = pd.read_csv(HADISD_STATIONS_URL)

    # check that data passed is a DataFrame object
    if type(df) != pd.DataFrame:
        raise ValueError(
            "The function requires a pandas DataFrame object as the data input"
        )

    # normalize simulation column name
    if "simulation" in df.columns and "sim" not in df.columns:
        df = df.rename(columns={"simulation": "sim"})

    # size check on TMY dataframe
    df = _tmy_8760_size_check(df)

    # Normalize time format: fix functions in _tmy_8760_size_check may
    # convert time to datetime objects (with seconds).  Re-format to
    # consistent "%Y-%m-%d %H:%M" strings so downstream writers and
    # _tmy_reset_time_for_gwl see a uniform format.
    df["time"] = pd.to_datetime(df["time"]).dt.strftime("%Y-%m-%d %H:%M")

    def _utc_offset_timezone(lat, lon):
        """Based on user input of lat lon, returns the UTC offset for that timezone

        Parameters
        ----------
        lat : float
            latitude of point of interest
        lon : float
            longitude of point of interest

        Returns
        -------
        str

        Modified from:
        https://stackoverflow.com/questions/5537876/get-utc-offset-from-time-zone-name-in-python

        """
        tf = TimezoneFinder()
        tzn = tf.timezone_at(lng=lon, lat=lat)

        time_now = datetime.datetime.now(pytz.timezone(tzn))
        tz_offset = time_now.utcoffset().total_seconds() / 60 / 60

        diff = "{:d}".format(int(tz_offset))

        return diff

    # custom location input handling
    match station_code:
        case str():  # custom code passed
            station_code = station_code
            state = stn_state
            timezone = _utc_offset_timezone(lon=stn_lon, lat=stn_lat)
            elevation = (
                stn_elev  # default of 0.0 on custom inputs if elevation is not provided
            )

        case int():  # hadisd station code passed
            # look up info
            if station_code in station_df["station id"].values:
                state = station_df.loc[station_df["station id"] == station_code][
                    "state"
                ].values[0]
                elevation = station_df.loc[station_df["station id"] == station_code][
                    "elevation"
                ].values[0]
                station_code = str(station_code)[:6]
                timezone = _utc_offset_timezone(lon=stn_lon, lat=stn_lat)
        case _:
            raise ValueError("station_code needs to be either str or int")

    def _tmy_header(
        location_name: str,
        station_code: int,
        stn_lat: float,
        stn_lon: float,
        state: str,
        timezone: str,
        elevation: float,
        df: pd.DataFrame,
    ) -> list[str]:
        """Constructs the header for the TMY output file in .tmy format

        Parameters
        ----------
        location_name : str
        station_code : int
        stn_lat : float
        stn_lon : float
        state : str
        timezone : str
        elevation : float
        df : pd.DataFrame

        Returns
        -------
        headers : list[str]

        Source: https://www.nrel.gov/docs/fy08osti/43156.pdf (pg. 3)

        """
        # line 1 - site information
        # line 1: USAF, station name quote delimited, state, time zone, lat, lon, elev (m)
        line_1 = "{0},'{1}',{2},{3},{4},{5},{6},{7}\n".format(
            station_code,
            location_name,
            state,
            timezone,
            stn_lat,
            stn_lon,
            elevation,
            df["sim"].values[0],
        )

        # line 2 - data field name and units, manually setting to ensure matches TMY3 labeling
        line_2 = (
            ",".join(
                [
                    "Air temperature at 2m (degC)",
                    "Dew point temperature at 2m (degC)",
                    "Relative humidity (0-100)",
                    "Instantaneous downwelling shortwave flux at bottom (W/m2)",
                    "Shortwave surface downward direct normal irradiance (W/m2)",
                    "Shortwave surface downward diffuse irradiance (W/m2)",
                    "Instantaneous downwelling longwave flux at bottom (W/m2)",
                    "Wind speed at 10m (m/s)",
                    "Wind direction at 10m (degrees)",
                    "Surface pressure (Pa)",
                ]
            )
            + "\n"
        )

        headers = [line_1, line_2]

        return headers

    def _epw_header(
        location_name: str,
        station_code: int,
        stn_lat: float,
        stn_lon: float,
        state: str,
        timezone: str,
        elevation: float,
        years: Tuple[int, int],
        df: pd.DataFrame,
    ) -> list[str]:
        """Constructs the header for the TMY output file in .epw format

        Parameters
        ----------
        location_name : str
        station_code : int
        stn_lat : float
        stn_lon : float
        state : str
        timezone : str
        elevation : float
        df : pd.DataFrame

        Returns
        -------
        headers : list[str]

        Source: EnergyPlus Version 23.1.0 Documentation

        """
        # line 1 - location, location name, state, country, WMO, lat, lon
        # line 1 - location, location name, state, country, weather station number (2 cols), lat, lon, time zone, elevation
        line_1 = f"LOCATION,{location_name.upper()},{state},USA,{'Custom_{}'.format(station_code)},{station_code},{stn_lat},{stn_lon},{timezone},{elevation}\n"

        # line 2 - design conditions, leave blank for now
        line_2 = "DESIGN CONDITIONS\n"

        # line 3 - typical/extreme periods, leave blank for now
        line_3 = "TYPICAL/EXTREME PERIODS\n"

        # line 4 - ground temperatures, leave blank for now
        line_4 = "GROUND TEMPERATURES\n"

        # line 5 - holidays/daylight savings, leap year (yes/no), daylight savings start, daylight savings end, num of holidays
        line_5 = "HOLIDAYS/DAYLIGHT SAVINGS,No,0,0,0\n"

        if "warming_level" in df.columns:
            warming_level = df["warming_level"].values[0]
            simulation = df["sim"].values[0]
            # line 6 - comments 1, going to include simulation + warming level information here
            line_6 = f"COMMENTS 1,TMY data produced on the Cal-Adapt: Analytics Engine, Warming Level: {warming_level}{degree_sign}C, Simulation: {simulation}\n"
            # line 7 - comments 2, including date range here from which TMY calculated
            line_7 = f"COMMENTS 2,TMY data produced using {warming_level}{degree_sign}C warming level. Year corresponds to index (1-30) in 30-year window centered on warming level. Model years for {warming_level}{degree_sign}C warming level in simulation {simulation} are {years[0]}-{years[1]}\n"
        else:
            # line 6 - comments 1, going to include simulation + scenario information here
            if "scenario" in df.columns:
                # get_data approach has a separate scenario column
                # the scenario is not included in the simulation name
                scenario = df["scenario"].values[0]
                line_6 = f"COMMENTS 1,TMY data produced on the Cal-Adapt: Analytics Engine, Simulation: {df['sim'].values[0]}, Scenario: {scenario}\n"
            else:
                # new core approach does not have a separate scenario column, scenario is included in simulation name
                # scenario information is included in the simulation name
                line_6 = f"COMMENTS 1,TMY data produced on the Cal-Adapt: Analytics Engine, Simulation: {df['sim'].values[0]}\n"
            # line 7 - comments 2, including date range here from which TMY calculated
            line_7 = f"COMMENTS 2,TMY data produced using {years[0]}-{years[1]} climatological period\n"

        # line 8 - data periods, num data periods, num records per hour, data period name, data period start day of week, data period start (Jan 1), data period end (Dec 31)
        line_8 = "DATA PERIODS,1,1,Data,,1/ 1,12/31\n"

        headers = [line_1, line_2, line_3, line_4, line_5, line_6, line_7, line_8]

        return headers

    # typical meteorological year format
    match file_ext:
        case "tmy":
            path_to_file = filename_to_export + ".tmy"

            with open(path_to_file, "w") as f:
                f.writelines(
                    _tmy_header(
                        location_name,
                        station_code,
                        stn_lat,
                        stn_lon,
                        state,
                        timezone,
                        elevation,
                        df,
                    )
                )  # writes required header lines
                # Keep only time + the 10 TMY variables, in the order
                # that matches the line-2 header written by _tmy_header.
                tmy_data_cols = [
                    "time",
                    "Air Temperature at 2m",
                    "Dew point temperature",
                    "Relative humidity",
                    "Instantaneous downwelling shortwave flux at bottom",
                    "Shortwave surface downward direct normal irradiance",
                    "Shortwave surface downward diffuse irradiance",
                    "Instantaneous downwelling longwave flux at bottom",
                    "Wind speed at 10m",
                    "Wind direction at 10m",
                    "Surface Pressure",
                ]
                df = df[tmy_data_cols]
                dfAsString = df.to_csv(sep=",", header=False, index=False)
                f.write(dfAsString)  # writes data in TMY format
            print(
                f"TMY data exported to .tmy format with filename {path_to_file}, with size {len(df)}"
            )
        # energy plus weather format
        case "epw":
            path_to_file = filename_to_export + ".epw"
            with open(path_to_file, "w") as f:
                f.writelines(
                    _epw_header(
                        location_name,
                        station_code,
                        stn_lat,
                        stn_lon,
                        state,
                        timezone,
                        elevation,
                        years,
                        df,
                    )
                )  # writes required header lines
                # WL time change happens in _epw_format_data if needed
                df_string = _epw_format_data(df).to_csv(
                    sep=",", header=False, index=False
                )
                f.write(df_string)  # writes data in EPW format
            print(
                f"TMY data exported to .epw format with filename {filename_to_export}, with size {len(df)}"
            )
        case "csv":
            #columns = [
            #     "index",
            #     "simulation",
            #     "time",
            #     "lat",
            #     "lon",
            #     "Air Temperature at 2m",
            #     "Dew point temperature",
            #     "Relative humidity",
            #     "Instantaneous downwelling shortwave flux at bottom",
            #     "Shortwave surface downward direct normal irradiance",
            #     "Shortwave surface downward diffuse irradiance ()",
            #     "Instantaneous downwelling longwave flux at bottom ()",
            #     "Wind speed at 10m ()",
            #     "Wind direction at 10m ()",
            #     "Surface Pressure (Pa)",
            # ]
            columns = ['simulation', 'time', 'lat', 'lon', 'Air temperature at 2m (degC)',
       'Dew point temperature at 2m (degC)', 'Relative humidity (0-100)',
       'Instantaneous downwelling shortwave flux at bottom (W/m2)',
       'Shortwave surface downward direct normal irradiance (W/m2)',
       'Shortwave surface downward diffuse irradiance (W/m2)',
       'Instantaneous downwelling longwave flux at bottom (W/m2)',
       'Wind speed at 10m (m/s)', 'Wind direction at 10m (degrees)',
       'Surface pressure (Pa)']
            print(df.columns)

            if "warming_level" in df.columns:
                df["centered_year"] = pd.to_numeric(
                    df["centered_year"], downcast="integer"
                )
                # set position of GWL specific columns
                columns.insert(3, "warming_level")
                columns.insert(6, "centered_year")
            # else:
            #     # set order of scenario column
            #     columns.insert(5, "scenario")
            df = df.rename(columns={"sim": "simulation"})
            df = df[columns]
            path_to_file = filename_to_export + ".csv"
            df.to_csv(path_to_file, index=False)
            print(
                f"TMY data exported to .csv format with filename {filename_to_export}, with size {len(df)}"
            )
        case _:
            print(
                'Please pass either "tmy","epw", or "csv" as a file format for export.'
            )

In [ ]:
for sim, xmy in xmy_data_to_export.items():
    filename = "temp_{0}_shock_xmy_{1}_{2}".format(
        extreme,stn_name.replace(" ", "_").replace("(", "").replace(")", ""), sim
    ).lower()
    write_tmy_file(
        filename,
        xmy_data_to_export[sim],
        (start_year, end_year),
        stn_name,
        stn_code,
        stn_lat,
        stn_lon,
        stn_state,
        file_ext="csv", #epw
    )